# VAD Downstream 実験ノートブック

このノートブックでは `vad_downstream/` のモジュールを1ステップずつ呼び出しながら実験を進めます。

**実行順序**
1. 環境セットアップ
2. データ読み込み・確認
3. モデル構築・確認
4. Stage 1 学習（VA-CCC）
5. Stage 2 学習（感情分類）
6. 結果まとめ・可視化

## 1. 環境セットアップ

In [ ]:
import sys, os

# vad_downstream/ をモジュール検索パスに追加（このノートブックが vad_downstream/ 内にある場合）
sys.path.insert(0, os.path.dirname(os.path.abspath("__file__")))

import torch
import numpy as np
import matplotlib.pyplot as plt

from model import EmotionClassifier
from loss import stage1_loss, ccc_loss
from data import load_iemocap_with_va, build_dataloaders
from train import train_stage1, eval_stage1, train_stage2, evaluate

print("PyTorch:", torch.__version__)
print("GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "なし（CPUで実行）")
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

## 2. パス設定

**ここだけ自分の環境に合わせて書き換えてください。**

In [ ]:
# ---- ここを自分の環境に合わせて書き換える ----
FEAT_PATH = "/path/to/iemocap_feats"   # .npy / .lengths / .emo ファイルの共通パス（拡張子なし）
VA_PATH   = "/path/to/va_labels.txt"   # utterance_id valence arousal の3列ファイル
BATCH_SIZE = 32
# ------------------------------------------

LABEL_DICT = {"ang": 0, "hap": 1, "neu": 2, "sad": 3}
N_SAMPLES  = [1085, 1023, 1151, 1031, 1241]  # Session 1〜5 のサンプル数

print("FEAT_PATH:", FEAT_PATH)
print("VA_PATH  :", VA_PATH)

## 3. データ読み込み・確認

In [ ]:
data = load_iemocap_with_va(FEAT_PATH, LABEL_DICT, VA_PATH)

print("サンプル数      :", data["num"])
print("特徴量の形状    :", data["feats"].shape, "  ← (全フレーム数, 768)")
print("VAラベルの形状  :", data["va_labels"].shape, "  ← (サンプル数, 2) [Valence, Arousal]")
print()
print("Valence の範囲  :", data["va_labels"][:, 0].min(), "〜", data["va_labels"][:, 0].max())
print("Arousal の範囲  :", data["va_labels"][:, 1].min(), "〜", data["va_labels"][:, 1].max())
print()
print("カテゴリラベルの内訳:")
for name, idx in LABEL_DICT.items():
    count = sum(1 for l in data["labels"] if l == idx)
    print(f"  {name}: {count} 件")

In [ ]:
# VA値の分布をグラフで確認
fig, axes = plt.subplots(1, 2, figsize=(10, 4))

axes[0].hist(data["va_labels"][:, 0], bins=30, color="steelblue")
axes[0].set_title("Valence 分布")
axes[0].set_xlabel("Valence 値")
axes[0].set_ylabel("頻度")

axes[1].hist(data["va_labels"][:, 1], bins=30, color="coral")
axes[1].set_title("Arousal 分布")
axes[1].set_xlabel("Arousal 値")

plt.tight_layout()
plt.show()

## 4. モデル構築・確認

In [ ]:
model = EmotionClassifier(
    input_dim=768,
    hidden_dim=256,
    vad_dim=3,
    num_classes=len(LABEL_DICT),
).to(device)

print(model)
print()
total = sum(p.numel() for p in model.parameters())
print(f"総パラメータ数: {total:,}")

In [ ]:
# ダミー入力で forward の動作確認（実データがなくても形状チェックできる）
dummy_feats = torch.randn(4, 50, 768).to(device)   # (バッチ4, フレーム50, 特徴量768)
dummy_mask  = torch.zeros(4, 50, dtype=torch.bool).to(device)  # パディングなし

vad_out, logits_out = model(dummy_feats, dummy_mask)
print("VAD出力の形状    :", vad_out.shape,    "  ← (バッチ, 3) [V, A, D]")
print("ロジットの形状   :", logits_out.shape, "  ← (バッチ, 感情クラス数)")
print("VAD値の範囲（Tanh）:", vad_out.min().item(), "〜", vad_out.max().item())

## 5. DataLoader 作成（Fold 1 を例に）

In [ ]:
FOLD = 0  # 0〜4 で変更可能（0 = Session1 をテストセットに使う）

test_start = sum(N_SAMPLES[:FOLD])
test_end   = test_start + N_SAMPLES[FOLD]

train_loader, val_loader, test_loader = build_dataloaders(
    data, BATCH_SIZE, test_start, test_end, eval_is_test=False
)

print(f"Fold {FOLD + 1}: テスト範囲 [{test_start}, {test_end})")
print(f"  train バッチ数: {len(train_loader)}")
print(f"  val   バッチ数: {len(val_loader)}")
print(f"  test  バッチ数: {len(test_loader)}")

# バッチの中身を1つ確認
batch = next(iter(train_loader))
print()
print("バッチの中身:")
print("  feats の形状        :", batch["net_input"]["feats"].shape,    "  ← (バッチ, 最大フレーム数, 768)")
print("  padding_mask の形状 :", batch["net_input"]["padding_mask"].shape)
print("  labels の形状       :", batch["labels"].shape)
print("  va_labels の形状    :", batch["va_labels"].shape, "  ← (バッチ, 2) [V, A]")

## 6. Stage 1 学習（VA-CCC損失）

In [ ]:
STAGE1_EPOCHS = 30
STAGE1_LR     = 1e-3

# モデルをリセット（何度でも最初からやり直せる）
model = EmotionClassifier(input_dim=768, hidden_dim=256, vad_dim=3, num_classes=len(LABEL_DICT)).to(device)
opt1  = torch.optim.Adam(model.vad_decoder.parameters(), lr=STAGE1_LR)

s1_train_losses = []
s1_val_losses   = []

for epoch in range(STAGE1_EPOCHS):
    train_loss = train_stage1(model, train_loader, opt1, device)
    val_loss   = eval_stage1(model, val_loader, device)

    s1_train_losses.append(train_loss / len(train_loader))
    s1_val_losses.append(val_loss)

    if (epoch + 1) % 5 == 0:
        print(f"Epoch {epoch+1:3d} | train loss: {s1_train_losses[-1]:.4f} | val loss: {s1_val_losses[-1]:.4f}")

In [ ]:
# Stage 1 の損失曲線
plt.figure(figsize=(8, 4))
plt.plot(s1_train_losses, label="train loss")
plt.plot(s1_val_losses,   label="val loss")
plt.xlabel("Epoch")
plt.ylabel("CCC損失 (1 - CCC)")
plt.title("Stage 1: VA-CCC損失の推移")
plt.legend()
plt.grid(True)
plt.tight_layout()
plt.show()

print(f"最終 val CCC損失: {s1_val_losses[-1]:.4f}  （0 に近いほど良い）")

# Stage 1 の最良モデルを保存
torch.save(model.state_dict(), "stage1_best.pth")
print("stage1_best.pth を保存しました")

## 7. Stage 2 学習（感情分類）

In [ ]:
STAGE2_EPOCHS  = 30
STAGE2_LR_CLS  = 1e-3   # 線形分類器の学習率
STAGE2_LR_FNN  = 1e-4   # VADDecoder（FNN）の学習率（Stage1 の 1/10）

# Stage1 の重みを引き継ぐ
model.load_state_dict(torch.load("stage1_best.pth"))

opt2 = torch.optim.Adam([
    {"params": model.vad_decoder.parameters(), "lr": STAGE2_LR_FNN},
    {"params": model.classifier.parameters(),  "lr": STAGE2_LR_CLS},
])
criterion = torch.nn.CrossEntropyLoss()

s2_train_losses = []
s2_val_wa_list  = []

best_val_wa = 0.0

for epoch in range(STAGE2_EPOCHS):
    train_loss = train_stage2(model, train_loader, opt2, criterion, device)
    val_wa, val_ua, val_f1 = evaluate(model, val_loader, device, num_classes=len(LABEL_DICT))

    s2_train_losses.append(train_loss / len(train_loader))
    s2_val_wa_list.append(val_wa)

    if val_wa > best_val_wa:
        best_val_wa = val_wa
        torch.save(model.state_dict(), "stage2_best.pth")

    if (epoch + 1) % 5 == 0:
        print(f"Epoch {epoch+1:3d} | loss: {s2_train_losses[-1]:.4f} | val WA: {val_wa:.2f}%  UA: {val_ua:.2f}%  F1: {val_f1:.2f}%")

In [ ]:
# Stage 2 の損失・精度曲線
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

axes[0].plot(s2_train_losses, color="steelblue")
axes[0].set_title("Stage 2: 訓練損失の推移")
axes[0].set_xlabel("Epoch")
axes[0].set_ylabel("CrossEntropy 損失")
axes[0].grid(True)

axes[1].plot(s2_val_wa_list, color="coral")
axes[1].set_title("Stage 2: 検証 WA の推移")
axes[1].set_xlabel("Epoch")
axes[1].set_ylabel("WA (%)")
axes[1].grid(True)

plt.tight_layout()
plt.show()

## 8. テスト評価・結果まとめ

In [ ]:
# 最良チェックポイントでテスト評価
model.load_state_dict(torch.load("stage2_best.pth"))
test_wa, test_ua, test_f1 = evaluate(model, test_loader, device, num_classes=len(LABEL_DICT))

print(f"=== Fold {FOLD + 1} テスト結果 ===")
print(f"  WA  : {test_wa:.2f}%")
print(f"  UA  : {test_ua:.2f}%")
print(f"  F1  : {test_f1:.2f}%")

In [ ]:
# VAD空間の可視化: テストセットの発話がどこにプロットされるか確認
model.eval()
all_vad = []
all_labels = []

with torch.no_grad():
    for batch in test_loader:
        feats        = batch["net_input"]["feats"].to(device)
        padding_mask = batch["net_input"]["padding_mask"].to(device)
        labels       = batch["labels"]

        vad, _ = model(feats, padding_mask)
        all_vad.append(vad.cpu().numpy())
        all_labels.append(labels.numpy())

all_vad    = np.concatenate(all_vad,    axis=0)  # (N, 3)
all_labels = np.concatenate(all_labels, axis=0)  # (N,)

label_names  = list(LABEL_DICT.keys())
colors       = ["red", "green", "blue", "orange"]

fig, axes = plt.subplots(1, 3, figsize=(15, 4))
pairs = [("Valence", "Arousal", 0, 1), ("Valence", "Dominance", 0, 2), ("Arousal", "Dominance", 1, 2)]

for ax, (xlabel, ylabel, xi, yi) in zip(axes, pairs):
    for cls_idx, (name, color) in enumerate(zip(label_names, colors)):
        mask = all_labels == cls_idx
        ax.scatter(all_vad[mask, xi], all_vad[mask, yi], label=name, color=color, alpha=0.4, s=10)
    ax.set_xlabel(xlabel)
    ax.set_ylabel(ylabel)
    ax.set_title(f"{xlabel} vs {ylabel}")
    ax.legend(markerscale=2)
    ax.grid(True)

plt.suptitle("テストセット: VAD空間での感情分布", y=1.02)
plt.tight_layout()
plt.show()